## Example 1: Local timezone detection and datetime handling

In [ ]:
import tzlocal
from datetime import datetime

# Example 1: Local timezone detection and datetime handling
print("Example 1: Local timezone detection and datetime handling")

try:
    # Get the local timezone
    local_tz = tzlocal.get_localzone()
    print(f"Local timezone: {local_tz}")
    print(f"Timezone name: {local_tz.key if hasattr(local_tz, 'key') else str(local_tz)}")
    
    # Create datetime objects with local timezone
    now_naive = datetime.now()
    now_local = local_tz.localize(now_naive) if hasattr(local_tz, 'localize') else now_naive.replace(tzinfo=local_tz)
    
    print(f"Naive datetime: {now_naive}")
    print(f"Localized datetime: {now_local}")
    print(f"UTC offset: {now_local.utcoffset()}")
    assert now_local.utcoffset() is not None
    
    # Test timezone-aware operations
    utc_time = now_local.utctimetuple()
    print(f"UTC time tuple: {utc_time[:6]}")  # Show first 6 elements (year, month, day, hour, min, sec)
    
    print("tzlocal example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"tzlocal example 1 failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Local zone name and zone object

In [ ]:
import tzlocal
from datetime import datetime, tzinfo

# Example 2: Local zone name and zone object
# get_localzone_name() returns the configured IANA name, or None when the machine has no
# time zone configuration; get_localzone() then falls back to /etc/localtime or UTC.
name = tzlocal.get_localzone_name()
zone = tzlocal.get_localzone()
print(f"get_localzone_name(): {name!r}")
print(f"get_localzone().key:  {getattr(zone, 'key', None)!r}")

assert name is None or isinstance(name, str), name
assert isinstance(zone, tzinfo), type(zone)
assert datetime.now(zone).utcoffset() is not None
# The zone is looked up once and cached.
assert tzlocal.get_localzone() is zone

## Example 3: Overriding the zone with the TZ environment variable

In [ ]:
import os
import tzlocal
from datetime import datetime, timedelta

# Example 3: Overriding the zone with the TZ environment variable
# tzlocal reads TZ first; reload_localzone() clears its cache so a change is picked up.
# Needs the IANA time zone database (the system zoneinfo files or the tzdata package).
saved_tz = os.environ.get('TZ')
try:
    os.environ['TZ'] = 'Asia/Tokyo'
    zone = tzlocal.reload_localzone()
    print(f"TZ=Asia/Tokyo -> name {tzlocal.get_localzone_name()!r}, key {zone.key!r}")
    assert zone.key == 'Asia/Tokyo', zone.key
    assert tzlocal.get_localzone_name() == 'Asia/Tokyo'
    assert tzlocal.get_localzone() is zone
    # Japan has no daylight saving time, so the offset is +09:00 all year.
    assert datetime(2024, 1, 15, 12, tzinfo=zone).utcoffset() == timedelta(hours=9)
    assert datetime(2024, 7, 15, 12, tzinfo=zone).utcoffset() == timedelta(hours=9)
finally:
    if saved_tz is None:
        os.environ.pop('TZ', None)
    else:
        os.environ['TZ'] = saved_tz
    tzlocal.reload_localzone()
print(f"restored local zone: {tzlocal.get_localzone()}")

## Example 4: Converting between the local zone and UTC

In [ ]:
import tzlocal
from datetime import datetime, timezone

# Example 4: Converting between the local zone and UTC
local = tzlocal.get_localzone()
now_local = datetime.now(local)
now_utc = now_local.astimezone(timezone.utc)
print(f"local: {now_local.isoformat()}")
print(f"UTC:   {now_utc.isoformat()}")

# Both are the same instant, and converting back gives the original wall-clock time.
assert now_local == now_utc
assert now_utc.astimezone(local).replace(tzinfo=None) == now_local.replace(tzinfo=None)
assert now_local.utcoffset() == now_local.replace(tzinfo=None) - now_utc.replace(tzinfo=None)

## Example 5: Windows and IANA zone name mappings

In [ ]:
from tzlocal import windows_tz

# Example 5: Windows and IANA zone name mappings
# tzlocal ships the Windows <-> IANA name tables it uses on Windows; they are plain dicts.
print(f"'Tokyo Standard Time' -> {windows_tz.win_tz['Tokyo Standard Time']}")
print(f"'Europe/Berlin' -> {windows_tz.tz_win['Europe/Berlin']}")
assert windows_tz.win_tz['Tokyo Standard Time'] == 'Asia/Tokyo'
assert windows_tz.tz_win['Asia/Tokyo'] == 'Tokyo Standard Time'
assert windows_tz.tz_win['Europe/Berlin'] == 'W. Europe Standard Time'